# Experiment 10 — J-space: abstraction vs translation

**Question.** Experiment 3 found law identity is strongly *readable* mid-stack. Experiment 1
found that steering the story→literal direction closes **0%** of the no-think gap. So: is the
hard part of Story→Rigid Grammar a deliberate, verbalizable computation (abstraction), while
emitting RG syntax is more automatic (translation)?

The [J-space / Jacobian lens paper](https://transformer-circuits.pub/2026/workspace/index.html)
gives a handle on that split. J-space is the set of residual-stream directions tied to tokens
the model is disposed to verbalize — a candidate "workspace" for flexible internal reasoning.
Ablating it hurts multi-hop reasoning while leaving fluent generation mostly intact.

This notebook runs three attached tests (not exploratory lens tourism):

| Part | What | Reads out if… |
|---|---|---|
| **A — Silent intermediates** | J-lens readout at the final prompt token on story vs literal-NL | Structure / math tokens rise mid-stack on stories that formalize correctly |
| **B — Ablation dissociation** | Zero top-k J-space directions in the workspace band during generation | Story→RG faithfulness drops more than literal→RG; validity mostly survives |
| **C — Law-identity swap** | Swap differentiating J-lens coordinates between two same-shape laws | Emitted ASSUME/ASK flips toward the swapped-in law |

**Model.** `Qwen/Qwen3-4B`, no-think, vendor sampling — same as experiments 1 / 3 / 8.
**Lens.** Pre-fitted Neuronpedia artifact for Qwen3-4B (`neuronpedia/jacobian-lens`).
**Runtime.** Colab GPU. Part A is one forward pass per prompt; B generates under hooks
(~slower); C is a small causal pilot. Everything checkpoints to Drive.

**Layer indexing.** This notebook uses **jlens block indices** (0-based decoder layers).
Experiment 3's `hidden_states[L]` is the output of block `L-1`, so Exp 3 layer 24 ≈ jlens 23.
We sweep jlens `{12, 18, 24, 30}` around that mid-stack peak.


In [ ]:
# ------------------------------ Configuration ------------------------------

REPO_URL = "https://github.com/shivam-raval96/semantic-drift-autoformalization.git"
REPO_COMMIT = "c9e128f247b6daff5b47fd9711ded5e02d1095e9"  # same pin as experiments 01 / 03 / 08

JLENS_REPO_URL = "https://github.com/anthropics/jacobian-lens.git"

MODEL_NAME = "Qwen/Qwen3-4B"
LENS_REPO = "neuronpedia/jacobian-lens"
LENS_FILE = "qwen3-4b/jlens/Salesforce-wikitext/Qwen3-4B_jacobian_lens.pt"

SEED = 0
PER_BIN = 5                 # stratified pairs per ops_total bin 1..8 (~40 after vacuous drop)
STORY_THEME = "paint"       # fixed theme so story/literal differ only in form

# jlens block indices (see intro). Ablation band is denser around the Exp-3 peak.
READOUT_LAYERS = [12, 18, 24, 30]
ABLATION_LAYERS = [16, 18, 20, 22, 24]
PRIMARY_LAYER = 24

ABLATION_K = 10             # top-k J-lens directions to project out per position
MAX_NEW_TOKENS = 256
BATCH_SIZE = 4              # ablation hooks are heavier; keep batches small

# Token families for Part A (single-token proxies; RG laws are multi-token).
STRUCTURE_WORDS = ["ASSUME", "ASK", "op", "operation", "equation", "equal", "function",
                   "algebra", "math", "variable", "implies", "identity"]
LITERAL_WORDS = ["operation", "Value", "input", "output", "apply", "result", "equals"]
NARRATIVE_WORDS = ["paint", "pigment", "pour", "workshop", "colorist", "Batch",
                   "crimson", "ochre", "apprentice"]

# Which parts to run (all True by default; flip off to skip).
RUN_PART_A = True
RUN_PART_B = True
RUN_PART_C = True

QUICK_TEST = True           # smoke-test plumbing first; set False for the full run
if QUICK_TEST:
    PER_BIN = 1
    READOUT_LAYERS = [18, 24]
    ABLATION_LAYERS = [20, 24]
    ABLATION_K = 5
    MAX_NEW_TOKENS = 128
    BATCH_SIZE = 2


In [ ]:
# ------------------------- Environment and outputs -------------------------
# Installs deps, clones the dataset/grader repo + jlens, mounts Drive for checkpoints.

%pip install -q -U "transformers>=4.51" accelerate matplotlib huggingface_hub
%pip install -q "git+https://github.com/anthropics/jacobian-lens.git"

import hashlib
import json
import math
import random
import subprocess
import sys
from collections import Counter, defaultdict
from contextlib import contextmanager, nullcontext
from pathlib import Path

if not Path("semantic-drift-autoformalization").exists():
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
subprocess.run(
    ["git", "-C", "semantic-drift-autoformalization", "checkout", "-q", REPO_COMMIT],
    check=True,
)
sys.path.insert(0, str(Path("semantic-drift-autoformalization/informalizing-etp").resolve()))

from benchmark import drop_vacuous, load_equations, sample_pairs_stratified, wrap_prompt
from checkform import grade
from storyform import render_story

import jlens
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

OUT_DIR = Path("exp10-outputs")
try:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/mech-interp-experiments/exp10-jspace")
except Exception as error:
    print(f"Google Drive not available ({error}); using local {OUT_DIR}/")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT_DIR.resolve())


In [ ]:
# ------------------------------- Load model + lens -------------------------------
# force_bos=False: we feed chat-templated strings, same as generation.

if torch.cuda.is_available():
    dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
else:
    dtype = torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=dtype, device_map="auto"
)
hf_model.eval()
model = jlens.from_hf(hf_model, tokenizer, force_bos=False)

lens = jlens.JacobianLens.from_pretrained(LENS_REPO, filename=LENS_FILE)
print(model)
print(lens)

missing = sorted(set(READOUT_LAYERS + ABLATION_LAYERS) - set(lens.source_layers))
assert not missing, f"lens missing layers {missing}; fitted={lens.source_layers[:5]}..{lens.source_layers[-1]}"
assert max(READOUT_LAYERS + ABLATION_LAYERS) < model.n_layers

DEVICE = model.input_device
# Cache Jacobians on device for interventions.
J_CACHE = {L: lens.jacobians[L].to(DEVICE) for L in sorted(set(READOUT_LAYERS + ABLATION_LAYERS))}
LM_WEIGHT = hf_model.lm_head.weight.detach().float()  # [vocab, d] on CPU; moved on demand
print(f"dtype={dtype}, device={DEVICE}, vocab={LM_WEIGHT.shape[0]}, d={model.d_model}")


## Dataset — matched story / literal-NL formalization prompts

Same seeded stratified sampler as experiments 1 / 3 / 8. Each pair is rendered once as a
themed story and once as literal-NL, then wrapped with the repo's no-think formalization
prompt. Both arms share `pair_id` and metadata, so grading is comparable.


In [ ]:
# ------------------------------ Build the dataset ------------------------------

equations, equations_sha = load_equations()
print(f"ETP equation list: {len(equations)} equations (sha256 {equations_sha[:12]})")

story_samples = drop_vacuous(sample_pairs_stratified(equations, PER_BIN, SEED, form="story"))
literal_samples = drop_vacuous(sample_pairs_stratified(equations, PER_BIN, SEED, form="literal"))
assert [s["pair_id"] for s in story_samples] == [s["pair_id"] for s in literal_samples]

# Re-render stories with a fixed theme so the only form difference is story vs literal.
records = []
for story_s, lit_s in zip(story_samples, literal_samples):
    meta = story_s["metadata"]
    story_text, story_meta = render_story(
        meta["equation_e"], meta["equation_f"], theme_key=STORY_THEME
    )
    # Rebuild story prompt with the fixed-theme text (literal prompt already correct).
    from checkform import build_prompt, PROMPT_PATH
    story_prompt = build_prompt({"story": story_text}, template_path=PROMPT_PATH)
    records.append({
        "pair_id": story_s["pair_id"],
        "ops_total": story_s["ops_total"],
        "depth": story_s["depth"],
        "metadata": story_meta,
        "story": {
            "form": "story",
            "text": story_text,
            "prompt": wrap_prompt(story_prompt, "off", "", "story"),
        },
        "literal": {
            "form": "literal",
            "text": lit_s["story"],
            "prompt": wrap_prompt(lit_s["prompt"], "off", "", "literal"),
        },
    })

print(f"{len(records)} matched pairs (theme={STORY_THEME})")
print(f"\n--- story tail ({records[0]['pair_id']}) ---")
print("..." + records[0]["story"]["prompt"][-400:])
print(f"\n--- literal tail ({records[0]['pair_id']}) ---")
print("..." + records[0]["literal"]["prompt"][-400:])

RUN_TAG = hashlib.sha256(json.dumps({
    "model": MODEL_NAME, "seed": SEED, "per_bin": PER_BIN, "theme": STORY_THEME,
    "quick": QUICK_TEST, "pairs": [r["pair_id"] for r in records],
}).encode()).hexdigest()[:12]
print("run tag:", RUN_TAG)


## Shared harness — chat formatting, generation, grading, J-lens helpers


In [ ]:
# ------------------------------ Shared harness ------------------------------

def build_chat(prompt_text: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )


def token_ids_for_words(words):
    """Map surface words to vocabulary ids (skip multi-token / missing)."""
    out = {}
    for w in words:
        for candidate in (w, " " + w, w.lower(), " " + w.lower()):
            ids = tokenizer.encode(candidate, add_special_tokens=False)
            if len(ids) == 1:
                out[w] = ids[0]
                break
    return out


STRUCTURE_IDS = token_ids_for_words(STRUCTURE_WORDS)
LITERAL_IDS = token_ids_for_words(LITERAL_WORDS)
NARRATIVE_IDS = token_ids_for_words(NARRATIVE_WORDS)
print("structure singles:", STRUCTURE_IDS)
print("literal singles:", LITERAL_IDS)
print("narrative singles:", NARRATIVE_IDS)


def jlens_vector(layer: int, token_id: int) -> torch.Tensor:
    """Direction in block-`layer` residual space for vocabulary token `token_id`.

    Paper: rows of W_U J_ℓ. With jlens transport h ↦ J @ h (as a column),
    v_t = W[t] @ J.
    """
    J = J_CACHE[layer]  # [d, d]
    w = LM_WEIGHT[token_id].to(DEVICE)  # [d]
    return w @ J  # [d]


def project_out(hidden: torch.Tensor, directions: list[torch.Tensor]) -> torch.Tensor:
    """Remove components along each direction (sequential, non-orthonormal)."""
    h = hidden
    for v in directions:
        v = v.to(device=h.device, dtype=h.dtype)
        v = v / (v.norm() + 1e-8)
        # h: [batch, seq, d] or [seq, d]
        coeff = (h * v).sum(dim=-1, keepdim=True)
        h = h - coeff * v
    return h


@torch.no_grad()
def generate_batch(prompt_texts, max_new_tokens=MAX_NEW_TOKENS, hook_cm=None):
    tokenizer.padding_side = "left"
    texts = [build_chat(p) for p in prompt_texts]
    encoded = tokenizer(texts, return_tensors="pt", padding=True).to(hf_model.device)
    sampling = dict(do_sample=True, temperature=0.7, top_p=0.8, top_k=20)
    ctx = hook_cm if hook_cm is not None else nullcontext()
    with ctx:
        output = hf_model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            **sampling,
        )
    completions = output[:, encoded["input_ids"].shape[1]:]
    return tokenizer.batch_decode(completions, skip_special_tokens=True)


def correct_rate(rows):
    return sum(r["status"] == "correct" for r in rows) / max(len(rows), 1)


def unparseable_rate(rows):
    return sum(r["status"] == "unparseable" for r in rows) / max(len(rows), 1)


def load_jsonl(path: Path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


def save_jsonl(path: Path, rows):
    with path.open("w") as fh:
        for row in rows:
            fh.write(json.dumps(row) + "\n")


def run_graded(name, items, hook_factory=None):
    """items: list of {pair_id, prompt, metadata, form, ...}. Cached jsonl."""
    path = OUT_DIR / f"{name}-{RUN_TAG}.jsonl"
    cached = load_jsonl(path)
    if len(cached) == len(items):
        print(f"[{name}] {len(cached)} cached, correct {correct_rate(cached):.1%}, "
              f"unparseable {unparseable_rate(cached):.1%}")
        return cached

    rows = []
    torch.manual_seed(SEED)
    with path.open("w") as fh:
        for i in tqdm(range(0, len(items), BATCH_SIZE), desc=name):
            batch = items[i:i + BATCH_SIZE]
            hooks = hook_factory(batch) if hook_factory is not None else None
            responses = generate_batch([b["prompt"] for b in batch], hook_cm=hooks)
            for item, response in zip(batch, responses):
                verdict = grade(response, item["metadata"])
                row = {
                    "pair_id": item["pair_id"],
                    "form": item["form"],
                    "ops_total": item.get("ops_total"),
                    "status": verdict["status"],
                    "transform": verdict.get("transform"),
                    "response": response,
                }
                rows.append(row)
                fh.write(json.dumps(row) + "\n")
                fh.flush()
    print(f"[{name}] {len(rows)} rows, correct {correct_rate(rows):.1%}, "
          f"unparseable {unparseable_rate(rows):.1%}")
    return rows


## Part A — Silent intermediates

Apply the J-lens at the **final prompt token** (pre-generation summary position) across
`READOUT_LAYERS`. For each form, record the reciprocal rank of structure / literal /
narrative proxy tokens. Then generate clean no-think RG and ask: do stories that end up
*correct* show higher mid-stack structure loading than stories that fail?


In [ ]:
# ------------------------------ Part A: J-lens readout ------------------------------

def reciprocal_rank(logits_1d: torch.Tensor, token_id: int) -> float:
    # logits_1d: [vocab]
    order = logits_1d.argsort(descending=True)
    rank = (order == token_id).nonzero(as_tuple=True)[0].item()  # 0-based
    return 1.0 / (rank + 1)


def family_score(logits_1d: torch.Tensor, id_map: dict) -> float:
    if not id_map:
        return 0.0
    return float(np.mean([reciprocal_rank(logits_1d, tid) for tid in id_map.values()]))


@torch.no_grad()
def readout_prompt(prompt_text: str, layers):
    chat = build_chat(prompt_text)
    lens_logits, model_logits, input_ids = lens.apply(
        model, chat, layers=layers, positions=[-1], max_seq_len=2048,
    )
    # lens_logits[L]: [1, vocab]
    per_layer = {}
    for L in layers:
        logits = lens_logits[L][0]
        top = logits.topk(8).indices.tolist()
        per_layer[L] = {
            "top_tokens": [tokenizer.decode([t]) for t in top],
            "structure_rr": family_score(logits, STRUCTURE_IDS),
            "literal_rr": family_score(logits, LITERAL_IDS),
            "narrative_rr": family_score(logits, NARRATIVE_IDS),
        }
    return per_layer


PART_A_PATH = OUT_DIR / f"partA-readouts-{RUN_TAG}.json"

if not RUN_PART_A:
    print("Part A skipped")
    part_a = []
elif PART_A_PATH.exists():
    part_a = json.loads(PART_A_PATH.read_text())
    print(f"Part A: loaded {len(part_a)} cached readouts")
else:
    part_a = []
    for rec in tqdm(records, desc="partA-readout"):
        for form in ("story", "literal"):
            arm = rec[form]
            layers_data = readout_prompt(arm["prompt"], READOUT_LAYERS)
            part_a.append({
                "pair_id": rec["pair_id"],
                "form": form,
                "ops_total": rec["ops_total"],
                "layers": {str(L): layers_data[L] for L in READOUT_LAYERS},
            })
    PART_A_PATH.write_text(json.dumps(part_a, indent=2))
    print("wrote", PART_A_PATH)

# Example top tokens at PRIMARY_LAYER
if part_a:
    ex = next(r for r in part_a if r["form"] == "story")
    print(f"\nExample story readout @ L{PRIMARY_LAYER} ({ex['pair_id']}):")
    print(" ", ex["layers"][str(PRIMARY_LAYER)]["top_tokens"])


In [ ]:
# -------------------- Part A: clean baselines + structure×accuracy --------------------

baseline_items = []
for rec in records:
    for form in ("story", "literal"):
        baseline_items.append({
            "pair_id": rec["pair_id"],
            "form": form,
            "ops_total": rec["ops_total"],
            "metadata": rec["metadata"],
            "prompt": rec[form]["prompt"],
        })

if RUN_PART_A or RUN_PART_B or RUN_PART_C:
    baseline_rows = run_graded("baseline", baseline_items)
else:
    baseline_rows = []

baseline_by_key = {(r["pair_id"], r["form"]): r for r in baseline_rows}

if part_a and baseline_rows:
    # Mean family RR by form × layer
    layers = READOUT_LAYERS
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
    for ax, family, title in zip(
        axes,
        ["structure_rr", "literal_rr", "narrative_rr"],
        ["structure proxies", "literal proxies", "narrative proxies"],
    ):
        for form, color in (("story", "C0"), ("literal", "C1")):
            ys = []
            for L in layers:
                vals = [r["layers"][str(L)][family]
                        for r in part_a if r["form"] == form]
                ys.append(float(np.mean(vals)))
            ax.plot(layers, ys, marker="o", label=form, color=color)
        ax.set_title(title)
        ax.set_xlabel("jlens layer")
        ax.legend()
    axes[0].set_ylabel("mean reciprocal rank")
    fig.suptitle("Part A — J-lens family loading at final prompt token")
    fig.tight_layout()
    fig_path = OUT_DIR / f"partA-family-curves-{RUN_TAG}.png"
    fig.savefig(fig_path, dpi=150)
    plt.show()
    print("saved", fig_path)

    # Structure loading vs correctness (stories only, primary layer)
    story_correct, story_wrong = [], []
    for r in part_a:
        if r["form"] != "story":
            continue
        status = baseline_by_key.get((r["pair_id"], "story"), {}).get("status")
        score = r["layers"][str(PRIMARY_LAYER)]["structure_rr"]
        (story_correct if status == "correct" else story_wrong).append(score)
    print(f"\nStory structure RR @ L{PRIMARY_LAYER}: "
          f"correct n={len(story_correct)} mean={np.mean(story_correct) if story_correct else float('nan'):.4f} | "
          f"not-correct n={len(story_wrong)} mean={np.mean(story_wrong) if story_wrong else float('nan'):.4f}")


## Part B — Ablation dissociation

At every token position, across `ABLATION_LAYERS`, project out the top-`ABLATION_K`
active J-lens directions — **except** tokens the clean model was about to say (protected
set from a clean prefill top-10, plus RG syntax tokens). Compare:

1. **clean** — already graded above  
2. **jspace-ablate** — workspace contents removed  
3. **random-ablate** — same number of random directions, norm-matched control  

**Predicted signature if abstraction ≠ translation:** story faithfulness drops under
J-space ablation more than literal; unparseable rate stays relatively flat (syntax is
automatic). Random ablation should hurt less or hurt both forms similarly.


In [ ]:
# ------------------------------ Part B: ablation hooks ------------------------------

# Always protect common RG / chat tokens so we do not just smash the output channel.
_PROTECT_WORDS = ["ASSUME", "ASK", "op", ":", "=", "(", ")", ",", "x", "y", "z",
                  "Value", "\n", "assistant"]
PROTECT_IDS = set(token_ids_for_words(_PROTECT_WORDS).values())
for tid in (tokenizer.eos_token_id, tokenizer.pad_token_id, tokenizer.bos_token_id):
    if tid is not None:
        PROTECT_IDS.add(tid)


@torch.no_grad()
def clean_protect_ids(prompt_text: str, k=10) -> set[int]:
    """Tokens in the model's top-k next-token dist at the final prompt position."""
    chat = build_chat(prompt_text)
    ids = tokenizer(chat, return_tensors="pt").input_ids.to(hf_model.device)
    out = hf_model(ids)
    top = out.logits[0, -1].topk(k).indices.tolist()
    return set(top) | PROTECT_IDS


def make_ablation_context(layers, k, protect_ids_per_batch, mode="jspace"):
    """Return a context manager that installs ablation hooks for one generate() call.

    mode='jspace': project out top-k J-lens vectors (by lens logit), minus protected.
    mode='random': project out k random unit directions (control).
    """
    @contextmanager
    def cm():
        handles = []

        def make_hook(layer):
            J = J_CACHE[layer]
            rng = torch.Generator(device=DEVICE)
            rng.manual_seed(SEED + 1000 * layer)

            def hook(module, args, output):
                hidden = output[0] if isinstance(output, tuple) else output
                # hidden: [batch, seq, d]
                bsz, seq, d = hidden.shape
                h = hidden
                # Transport + unembed at every position (batch-serial for clarity).
                new_batches = []
                for b in range(bsz):
                    hb = h[b]  # [seq, d]
                    if mode == "random":
                        dirs = []
                        for _ in range(k):
                            v = torch.randn(d, generator=rng, device=DEVICE, dtype=torch.float32)
                            dirs.append(v)
                        hb2 = project_out(hb.float(), dirs).to(hb.dtype)
                        new_batches.append(hb2)
                        continue

                    transported = (hb.float() @ J.T)  # [seq, d]
                    # Unembed via the real head (norm + lm_head).
                    logits = model.unembed(transported).float()  # [seq, vocab] on lm_head device
                    logits = logits.to(DEVICE)
                    protect = protect_ids_per_batch[b]
                    dirs = []
                    # Use last position's top-k as a cheap proxy for the whole seq's
                    # active workspace contents (paper uses per-position; this is
                    # the Colab-budget approximation). Recompute per position only
                    # when seq==1 (decode steps).
                    positions = range(seq) if seq == 1 else [seq - 1]
                    chosen = []
                    for pos in positions:
                        scores = logits[pos].clone()
                        if protect:
                            scores[list(protect)] = -1e9
                        top = scores.topk(k).indices.tolist()
                        chosen.extend(top)
                    # Unique, preserve order
                    seen = set()
                    for tid in chosen:
                        if tid in seen:
                            continue
                        seen.add(tid)
                        dirs.append(jlens_vector(layer, tid))
                        if len(dirs) >= k:
                            break
                    hb2 = project_out(hb.float(), dirs).to(hb.dtype)
                    new_batches.append(hb2)

                h_new = torch.stack(new_batches, dim=0)
                if isinstance(output, tuple):
                    return (h_new,) + tuple(output[1:])
                return h_new

            return hook

        for L in layers:
            handles.append(hf_model.model.layers[L].register_forward_hook(make_hook(L)))
        try:
            yield
        finally:
            for h in handles:
                h.remove()
    return cm()


def ablation_hook_factory(mode):
    def factory(batch):
        protect = [clean_protect_ids(item["prompt"]) for item in batch]
        return make_ablation_context(ABLATION_LAYERS, ABLATION_K, protect, mode=mode)
    return factory


if RUN_PART_B:
    jspace_rows = run_graded(
        "jspace-ablate", baseline_items, hook_factory=ablation_hook_factory("jspace")
    )
    random_rows = run_graded(
        "random-ablate", baseline_items, hook_factory=ablation_hook_factory("random")
    )
else:
    jspace_rows, random_rows = [], []
    print("Part B skipped")


In [ ]:
# ------------------------------ Part B: summary table + figure ------------------------------

def summarize(rows, form):
    sub = [r for r in rows if r["form"] == form]
    return {
        "n": len(sub),
        "correct": correct_rate(sub),
        "unparseable": unparseable_rate(sub),
        "wrong": sum(r["status"] == "wrong" for r in sub) / max(len(sub), 1),
    }


if RUN_PART_B and baseline_rows:
    conditions = {
        "clean": baseline_rows,
        "jspace-ablate": jspace_rows,
        "random-ablate": random_rows,
    }
    print(f"{'condition':18s} {'form':8s} {'n':>4s} {'correct':>8s} {'wrong':>8s} {'unparse':>8s}")
    summary = {}
    for cname, rows in conditions.items():
        summary[cname] = {}
        for form in ("story", "literal"):
            s = summarize(rows, form)
            summary[cname][form] = s
            print(f"{cname:18s} {form:8s} {s['n']:4d} {s['correct']:8.1%} {s['wrong']:8.1%} {s['unparseable']:8.1%}")

    # Dissociation deltas: story drop minus literal drop under jspace ablation
    story_drop = summary["clean"]["story"]["correct"] - summary["jspace-ablate"]["story"]["correct"]
    lit_drop = summary["clean"]["literal"]["correct"] - summary["jspace-ablate"]["literal"]["correct"]
    print(f"\nFaithfulness drop (clean − jspace-ablate): story {story_drop:+.1%}, literal {lit_drop:+.1%}")
    print(f"Dissociation (story drop − literal drop): {story_drop - lit_drop:+.1%}")
    print("Signature we want: story drop > literal drop, with unparseable rates staying flat.")

    fig, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(3)
    width = 0.35
    for i, form in enumerate(("story", "literal")):
        ys = [summary[c][form]["correct"] for c in ("clean", "jspace-ablate", "random-ablate")]
        ax.bar(x + (i - 0.5) * width, ys, width, label=form)
    ax.set_xticks(x)
    ax.set_xticklabels(["clean", "jspace-ablate", "random-ablate"])
    ax.set_ylabel("correct rate")
    ax.set_ylim(0, 1)
    ax.legend()
    ax.set_title("Part B — ablation dissociation")
    fig.tight_layout()
    fig_path = OUT_DIR / f"partB-ablation-{RUN_TAG}.png"
    fig.savefig(fig_path, dpi=150)
    plt.show()
    print("saved", fig_path)

    (OUT_DIR / f"partB-summary-{RUN_TAG}.json").write_text(json.dumps(summary, indent=2))


## Part C — Law-identity swap (pilot)

Causal check that workspace contents carry *which law*, not just "math-ish-ness".

For each source record, pick a **same-`ops_total` foil** with a different `pair_id`.
Read J-lens top tokens for both at `PRIMARY_LAYER`, take tokens that strongly favor the
foil over the source, and **swap lens coordinates** (subtract source projection / add foil
projection) across `ABLATION_LAYERS` during generation on the source story prompt.

Success signal: graded status moves toward the foil's metadata more often than a
random-token swap control. This is a small pilot — multi-token laws will not map cleanly
onto single J-lens rows, so treat a null here as inconclusive rather than decisive.


In [ ]:
# ------------------------------ Part C: lens-coordinate swap ------------------------------

def pick_foils(recs):
    by_ops = defaultdict(list)
    for r in recs:
        by_ops[r["ops_total"]].append(r)
    pairs = []
    for r in recs:
        candidates = [c for c in by_ops[r["ops_total"]] if c["pair_id"] != r["pair_id"]]
        if not candidates:
            continue
        # Deterministic foil choice
        foil = sorted(candidates, key=lambda c: c["pair_id"])[0]
        pairs.append((r, foil))
    return pairs


@torch.no_grad()
def discriminating_token_ids(source_prompt, foil_prompt, layer, k=4):
    """Token ids with largest (foil − source) lens logit at the final prompt token."""
    def logits(prompt):
        chat = build_chat(prompt)
        lens_logits, _, _ = lens.apply(
            model, chat, layers=[layer], positions=[-1], max_seq_len=2048,
        )
        return lens_logits[layer][0]

    diff = logits(foil_prompt) - logits(source_prompt)
    # Ignore protected / ultra-common tokens
    diff = diff.clone()
    if PROTECT_IDS:
        diff[list(PROTECT_IDS)] = -1e9
    return diff.topk(k).indices.tolist()


def make_swap_context(layers, token_pairs, alpha=1.0):
    """token_pairs: list of (source_tid, foil_tid) to exchange in lens coordinates."""
    @contextmanager
    def cm():
        handles = []

        def make_hook(layer):
            def hook(module, args, output):
                hidden = output[0] if isinstance(output, tuple) else output
                h = hidden.float()
                for src_tid, foil_tid in token_pairs:
                    v_s = jlens_vector(layer, src_tid)
                    v_t = jlens_vector(layer, foil_tid)
                    V = torch.stack([v_s, v_t], dim=1).to(h.device)  # [d, 2]
                    # Least-squares coords: c = V^+ h
                    # Solve V c ≈ h^T for each position… use pinv on the 2-col matrix.
                    Vp = torch.linalg.pinv(V)  # [2, d]
                    # h: [batch, seq, d]
                    c = torch.einsum("cd,bsd->bsc", Vp, h)  # [batch, seq, 2]
                    c_swap = c.clone()
                    c_swap[..., 0], c_swap[..., 1] = c[..., 1], c[..., 0]
                    delta = torch.einsum("bsc,cd->bsd", alpha * (c_swap - c), V.T)
                    h = h + delta
                h = h.to(hidden.dtype)
                if isinstance(output, tuple):
                    return (h,) + tuple(output[1:])
                return h
            return hook

        for L in layers:
            handles.append(hf_model.model.layers[L].register_forward_hook(make_hook(L)))
        try:
            yield
        finally:
            for h in handles:
                h.remove()
    return cm()


if RUN_PART_C:
    foil_pairs = pick_foils(records)
    if QUICK_TEST:
        foil_pairs = foil_pairs[: min(4, len(foil_pairs))]
    print(f"Part C: {len(foil_pairs)} source→foil pairs")

    swap_items = []
    swap_plans = []
    for src, foil in foil_pairs:
        tids = discriminating_token_ids(
            src["story"]["prompt"], foil["story"]["prompt"], PRIMARY_LAYER, k=4
        )
        # Pair each foil-favoring token with a source-favoring counterpart via reverse diff
        src_tids = discriminating_token_ids(
            foil["story"]["prompt"], src["story"]["prompt"], PRIMARY_LAYER, k=4
        )
        token_pairs = list(zip(src_tids, tids))
        swap_plans.append({
            "pair_id": src["pair_id"],
            "foil_id": foil["pair_id"],
            "token_pairs": [
                (tokenizer.decode([a]), tokenizer.decode([b])) for a, b in token_pairs
            ],
            "token_pair_ids": token_pairs,
        })
        swap_items.append({
            "pair_id": src["pair_id"],
            "foil_id": foil["pair_id"],
            "form": "story",
            "ops_total": src["ops_total"],
            "metadata": src["metadata"],
            "foil_metadata": foil["metadata"],
            "prompt": src["story"]["prompt"],
            "token_pair_ids": token_pairs,
        })

    (OUT_DIR / f"partC-plans-{RUN_TAG}.json").write_text(json.dumps([
        {**p, "token_pair_ids": [[int(a), int(b)] for a, b in p["token_pair_ids"]]}
        for p in swap_plans
    ], indent=2))
    print("sample swap plan:", swap_plans[0] if swap_plans else None)

    def swap_factory(batch):
        # One plan per item; batching heterogeneous swaps → run batch size 1 effectively
        assert len(batch) == 1
        return make_swap_context(ABLATION_LAYERS, batch[0]["token_pair_ids"], alpha=1.0)

    # Force batch size 1 for heterogeneous interventions
    _bs = BATCH_SIZE
    BATCH_SIZE = 1
    swap_rows = run_graded("law-swap", swap_items, hook_factory=swap_factory)
    BATCH_SIZE = _bs

    # Did any swap make the response grade correct under the FOIL metadata?
    foil_hits = 0
    for item, row in zip(swap_items, swap_rows):
        foil_verdict = grade(row["response"], item["foil_metadata"])
        row["foil_status"] = foil_verdict["status"]
        foil_hits += foil_verdict["status"] == "correct"
    save_jsonl(OUT_DIR / f"law-swap-{RUN_TAG}.jsonl", swap_rows)

    base_story = [baseline_by_key[(it["pair_id"], "story")] for it in swap_items
                  if (it["pair_id"], "story") in baseline_by_key]
    print(f"\nPart C on {len(swap_rows)} stories:")
    print(f"  clean correct (same items): {correct_rate(base_story):.1%}")
    print(f"  after swap, correct vs SOURCE metadata: {correct_rate(swap_rows):.1%}")
    print(f"  after swap, correct vs FOIL metadata:   {foil_hits / max(len(swap_rows), 1):.1%}")
    print("If foil-correct rises while source-correct falls, workspace tokens carry law identity.")
else:
    print("Part C skipped")


## Wrap-up

How to read the run:

- **Part A.** If structure/literal proxy RR is higher mid-stack on literal than story, and
  higher on correct stories than failed ones, the workspace is at least tracking the
  abstraction state correlationally.
- **Part B.** The headline is dissociation: `story_drop − literal_drop` under J-space
  ablation, with unparseable rates flat. Random ablation is the control.
- **Part C.** Soft causal evidence only — single-token proxies for whole ETP laws are lossy.
  A positive foil-hit rate is exciting; a null is not a kill.

After the Colab run: `File → Download .ipynb` (or share the Drive `exp10-jspace/` folder)
and we can write `10-jspace-abstraction-vs-translation-findings.md`.


In [ ]:
# ------------------------------ Persist run config ------------------------------

config = {
    "run_tag": RUN_TAG,
    "model": MODEL_NAME,
    "lens_file": LENS_FILE,
    "seed": SEED,
    "per_bin": PER_BIN,
    "quick_test": QUICK_TEST,
    "theme": STORY_THEME,
    "readout_layers": READOUT_LAYERS,
    "ablation_layers": ABLATION_LAYERS,
    "ablation_k": ABLATION_K,
    "n_pairs": len(records),
    "equations_sha": equations_sha,
    "parts": {"A": RUN_PART_A, "B": RUN_PART_B, "C": RUN_PART_C},
}
(OUT_DIR / f"run-config-{RUN_TAG}.json").write_text(json.dumps(config, indent=2))
print(json.dumps(config, indent=2))
print("\nDone. Artifacts in", OUT_DIR.resolve())
